# Structure editing operations (Package A, Feature A4)

This notebook demonstrates the structure_ops API: sampling a configuration
from an average structure, repeating it into a supercell, substituting
species, checking the composition, and displacing atoms.  It uses the same
COD GaAs CIF fixture as [06_interop.ipynb](06_interop.ipynb) and the
structure_ops functions from `discuspdf`.

See [docs/STRUCTURE_OPS.md](../docs/STRUCTURE_OPS.md) for the full
documentation.

In [ ]:
import numpy as np
from discuspdf import Lattice, Structure, repeat, substitute, displace, sample_configuration, check_composition, read_cif

## Read the COD GaAs CIF

In [ ]:
cif = read_cif("tests/fixtures/interop/gaas_cod_9008845.cif")
s_avg = cif.structure
print(s_avg)
assert s_avg.model == "average"

## Sample a configuration

In [ ]:
s_conf = sample_configuration(s_avg, seed=42).structure
print(s_conf)
assert s_conf.model == "configuration"
assert s_conf.n_atoms == 8

## Repeat 4×4×4

In [ ]:
s_rep = repeat(s_conf, (4, 4, 4)).structure
print(s_rep)
assert s_rep.n_atoms == 512

## Substitute 10% Ga → In

In [ ]:
s_sub = substitute(s_rep, old="Ga", new="In", seed=42, fraction=0.10).structure
n_in = int(np.sum(s_sub.species == "In"))
print(f"In: {n_in}, Ga: {int(np.sum(s_sub.species == 'Ga'))}")
assert n_in == 26

## check_composition

In [ ]:
check_composition(s_sub, {"Ga": 230, "In": 26, "As": 256})
print("OK")

## Displace σ=0.05 Å

In [ ]:
s_disp = displace(s_sub, seed=42, sigma=0.05).structure
print(s_disp)
assert s_disp.n_atoms == 512

## Summary

* `sample_configuration` draws a disordered configuration from an average
  structure's occupancies (deterministic with `seed`).
* `repeat` builds a supercell (4×4×4 -> 512 atoms).
* `substitute` replaces a fraction of one species with another; the total
  composition is verified with `check_composition`.
* `displace` adds Gaussian displacements (σ = 0.05 Å) to every atom while
  preserving the atom count.